[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-4/sub-graph.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239937-lesson-2-sub-graphs)

# 子图（Sub-graphs）

## 回顾

我们正在构建一个多 Agent 研究助手，将本课程的所有模块内容串联起来。

我们刚刚介绍了并行化，这是一个重要的 LangGraph 可控性主题。

## 目标

现在，我们[将介绍子图](https://docs.langchain.com/oss/python/langgraph/use-subgraphs)。

## 状态

子图允许你在图的不同部分创建和管理不同的状态。

这对于多 Agent 系统特别有用，其中每个 Agent 团队都有各自的状态。

让我们考虑一个简化的例子：

* 我有一个接受日志的系统
* 它由不同的 Agent 执行两个单独的子任务（汇总日志、查找故障模式）
* 我想在两个不同的子图中执行这两个操作。

最关键的是理解图之间如何通信！

简而言之，通信是**通过重叠的键来完成的**：

* 子图可以从父图访问 `docs`
* 父图可以从子图访问 `summary/failure_report`

![subgraph.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbb1abf89f2d847ee6f1ff_sub-graph1.png)

## 输入

让我们为将输入到图的日志定义一个 schema。

In [ ]:
%%capture --no-stderr
%pip install -U  langgraph

我们将使用 [LangSmith](https://docs.langchain.com/langsmith/home) 进行[追踪](https://docs.langchain.com/langsmith/observability-concepts)。

In [ ]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langchain-academy"

In [ ]:
from operator import add
from typing_extensions import TypedDict
from typing import List, Optional, Annotated

# 日志的结构
class Log(TypedDict):
    id: str
    question: str
    docs: Optional[List]
    answer: str
    grade: Optional[int]
    grader: Optional[str]
    feedback: Optional[str]

## 子图

这是故障分析子图，使用 `FailureAnalysisState`。

In [ ]:
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END

# 故障分析子图
class FailureAnalysisState(TypedDict):
    cleaned_logs: List[Log]
    failures: List[Log]
    fa_summary: str
    processed_logs: List[str]

class FailureAnalysisOutputState(TypedDict):
    fa_summary: str
    processed_logs: List[str]

def get_failures(state):
    """ 获取包含故障的日志 """
    cleaned_logs = state["cleaned_logs"]
    failures = [log for log in cleaned_logs if "grade" in log]
    return {"failures": failures}

def generate_summary(state):
    """ 生成故障摘要 """
    failures = state["failures"]
    # 添加函数: fa_summary = summarize(failures)
    fa_summary = "Poor quality retrieval of Chroma documentation."
    return {"fa_summary": fa_summary, "processed_logs": [f"failure-analysis-on-log-{failure['id']}" for failure in failures]}

fa_builder = StateGraph(state_schema=FailureAnalysisState,output_schema=FailureAnalysisOutputState)
fa_builder.add_node("get_failures", get_failures)
fa_builder.add_node("generate_summary", generate_summary)
fa_builder.add_edge(START, "get_failures")
fa_builder.add_edge("get_failures", "generate_summary")
fa_builder.add_edge("generate_summary", END)

graph = fa_builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

这是问题汇总子图，使用 `QuestionSummarizationState`。

In [ ]:
# 汇总子图
class QuestionSummarizationState(TypedDict):
    cleaned_logs: List[Log]
    qs_summary: str
    report: str
    processed_logs: List[str]

class QuestionSummarizationOutputState(TypedDict):
    report: str
    processed_logs: List[str]

def generate_summary(state):
    cleaned_logs = state["cleaned_logs"]
    # 添加函数: summary = summarize(generate_summary)
    summary = "Questions focused on usage of ChatOllama and Chroma vector store."
    return {"qs_summary": summary, "processed_logs": [f"summary-on-log-{log['id']}" for log in cleaned_logs]}

def send_to_slack(state):
    qs_summary = state["qs_summary"]
    # 添加函数: report = report_generation(qs_summary)
    report = "foo bar baz"
    return {"report": report}

qs_builder = StateGraph(QuestionSummarizationState,output_schema=QuestionSummarizationOutputState)
qs_builder.add_node("generate_summary", generate_summary)
qs_builder.add_node("send_to_slack", send_to_slack)
qs_builder.add_edge(START, "generate_summary")
qs_builder.add_edge("generate_summary", "send_to_slack")
qs_builder.add_edge("send_to_slack", END)

graph = qs_builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

## 将子图添加到父图中

现在，我们可以将它们整合在一起。

我们创建带有 `EntryGraphState` 的父图。

并且我们将子图作为节点添加！

```
entry_builder.add_node("question_summarization", qs_builder.compile())
entry_builder.add_node("failure_analysis", fa_builder.compile())
```

In [ ]:
# 入口图
class EntryGraphState(TypedDict):
    raw_logs: List[Log]
    cleaned_logs: Annotated[List[Log], add] # 将被两个子图使用
    fa_summary: str # 仅在故障分析子图中生成
    report: str # 仅在问题汇总子图中生成
    processed_logs:  Annotated[List[int], add] # 将在两个子图中生成

但是，如果 `cleaned_logs` 只是作为输入*进入*每个子图而没有被修改，为什么它还需要一个 reducer 呢？

```
cleaned_logs: Annotated[List[Log], add] # 将被两个子图使用
```

这是因为子图的输出状态将包含**所有键**，即使它们未被修改。

子图是并行运行的。

因为并行子图返回相同的键，所以需要一个像 `operator.add` 这样的 reducer 来组合来自每个子图的输入值。

但是，我们可以通过使用之前讨论过的另一个概念来解决这个问题。

我们可以简单地为每个子图创建一个输出状态 schema，并确保输出状态 schema 包含不同的键作为输出发布。

实际上，我们并不需要每个子图都输出 `cleaned_logs`。

In [ ]:
# 入口图
class EntryGraphState(TypedDict):
    raw_logs: List[Log]
    cleaned_logs: List[Log]
    fa_summary: str # 仅在故障分析子图中生成
    report: str # 仅在问题汇总子图中生成
    processed_logs:  Annotated[List[int], add] # 将在两个子图中生成

def clean_logs(state):
    # 获取日志
    raw_logs = state["raw_logs"]
    # 数据清洗 raw_logs -> docs
    cleaned_logs = raw_logs
    return {"cleaned_logs": cleaned_logs}

entry_builder = StateGraph(EntryGraphState)
entry_builder.add_node("clean_logs", clean_logs)
entry_builder.add_node("question_summarization", qs_builder.compile())
entry_builder.add_node("failure_analysis", fa_builder.compile())

entry_builder.add_edge(START, "clean_logs")
entry_builder.add_edge("clean_logs", "failure_analysis")
entry_builder.add_edge("clean_logs", "question_summarization")
entry_builder.add_edge("failure_analysis", END)
entry_builder.add_edge("question_summarization", END)

graph = entry_builder.compile()

from IPython.display import Image, display

# 将 xray 设置为 1 将显示嵌套图的内部结构
display(Image(graph.get_graph(xray=1).draw_mermaid_png()))

In [ ]:
# 虚拟日志
question_answer = Log(
    id="1",
    question="How can I import ChatOllama?",
    answer="To import ChatOllama, use: 'from langchain_community.chat_models import ChatOllama.'",
)

question_answer_feedback = Log(
    id="2",
    question="How can I use Chroma vector store?",
    answer="To use Chroma, define: rag_chain = create_retrieval_chain(retriever, question_answer_chain).",
    grade=0,
    grader="Document Relevance Recall",
    feedback="The retrieved documents discuss vector stores in general, but not Chroma specifically",
)

raw_logs = [question_answer,question_answer_feedback]
graph.invoke({"raw_logs": raw_logs})

## LangSmith

让我们查看 LangSmith 追踪：

https://smith.langchain.com/public/f8f86f61-1b30-48cf-b055-3734dfceadf2/r